# **Notebook01_Dataset Exploration**

### Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Library berhasil diimpor!")

Library berhasil diimpor!


## **1. Load Dataset**

In [2]:
from pathlib import Path

# Lokasi folder data mentah
DATA_DIR = Path("../data/raw")

# Check if folder exist
print("Folder exist:", DATA_DIR.exists())

Folder exist: True


In [3]:
# Cari seluruh file CSV
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Jumlah file CSV: {len(csv_files)}")

# Baca semua file ke dalam dictionary
datasets = {}

for file in csv_files:
    table_name = file.stem
    datasets[table_name] = pd.read_csv(file)
    print(f"{table_name}: {datasets[table_name].shape}")

Jumlah file CSV: 9
olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_orders_dataset: (99441, 8)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


## **2. Data Structures Exploration**

### Tables Summary

In [4]:
summary = []

for table_name, df in datasets.items():
    summary.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,table_name,rows,columns,missing_values,duplicate_rows
0,olist_customers_dataset,99441,5,0,0
1,olist_geolocation_dataset,1000163,5,0,261831
2,olist_order_items_dataset,112650,7,0,0
3,olist_order_payments_dataset,103886,5,0,0
4,olist_order_reviews_dataset,99224,7,145903,0
5,olist_orders_dataset,99441,8,4908,0
6,olist_products_dataset,32951,9,2448,0
7,olist_sellers_dataset,3095,4,0,0
8,product_category_name_translation,71,2,0,0


### Columns dtypes

In [5]:
for table_name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"TABEL: {table_name}")
    print("=" * 60)
    print("Kolom:", list(df.columns))
    print("\nTipe data:")
    print(df.dtypes)


TABEL: olist_customers_dataset
Kolom: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Tipe data:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

TABEL: olist_geolocation_dataset
Kolom: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Tipe data:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

TABEL: olist_order_items_dataset
Kolom: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Tipe data:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping

### Look for missing values

In [6]:
for table_name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\nTABEL: {table_name}")

    if missing.empty:
        print("No missing values.")
    else:
        print(missing.sort_values(ascending=False))


TABEL: olist_customers_dataset
No missing values.

TABEL: olist_geolocation_dataset
No missing values.

TABEL: olist_order_items_dataset
No missing values.

TABEL: olist_order_payments_dataset
No missing values.

TABEL: olist_order_reviews_dataset
review_comment_title      87656
review_comment_message    58247
dtype: int64

TABEL: olist_orders_dataset
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

TABEL: olist_products_dataset
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

TABEL: olist_sellers_dataset
No missing values.

TABEL: product_category_name_translation
No missing values.


## **3. Data Cleaning**

### **3.1 Handle Geolocation Duplicates**
Combine identical zipcode by using coorinates median

In [7]:
geo = datasets["olist_geolocation_dataset"]

# Duplikasi pada seluruh kolom
print("all columns duplicated:", geo.duplicated().sum())

# Jumlah kode pos unik
print("Unique zipcode:", geo["geolocation_zip_code_prefix"].nunique())


all columns duplicated: 261831
Unique zipcode: 19015


In [8]:
# Mengambil data geolocation
geo = datasets["olist_geolocation_dataset"].copy()

# Menghapus duplikasi identik
geo = geo.drop_duplicates().reset_index(drop=True)

# Membuat satu koordinat representatif per kode pos
geo_mapping = (
    geo.groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median"),
        city=("geolocation_city", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        state=("geolocation_state", lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    )
    .reset_index()
)

# Validasi hasil
print("Number of row in Geolocation table:", len(geo))
print("Number of unique zipcode:", geo_mapping["geolocation_zip_code_prefix"].nunique())
print("Duplicated zipcode:", geo_mapping["geolocation_zip_code_prefix"].duplicated().sum())

Number of row in Geolocation table: 738332
Number of unique zipcode: 19015
Duplicated zipcode: 0


In [9]:
geo_mapping

,geolocation_zip_code_prefix,latitude,longitude,city,state
0,1001,-23.549951,-46.634027,sao paulo,SP
1,1002,-23.548228,-46.635247,sao paulo,SP
2,1003,-23.548977,-46.635313,sao paulo,SP
3,1004,-23.549550,-46.634771,sao paulo,SP
4,1005,-23.549763,-46.636100,sao paulo,SP
...,...,...,...,...,...
19010,99960,-27.953797,-52.029641,charrua,RS
19011,99965,-28.179542,-52.035551,agua santa,RS
19012,99970,-28.343257,-51.875470,ciriaco,RS
19013,99980,-28.388342,-51.846871,david canabarro,RS


In [20]:
datasets["olist_geolocation_dataset"] = geo_mapping

### **3.2 Handle missing values in Orders table**

In [10]:
datasets["olist_orders_dataset"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


#### check order status

In [11]:
orders = datasets["olist_orders_dataset"]

# Distribusi status pesanan
print("Distribusi status pesanan:")
print(orders["order_status"].value_counts())

# Missing values berdasarkan status pesanan
missing_by_status = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_approved=("order_approved_at", lambda x: x.isna().sum()),
        missing_carrier=("order_delivered_carrier_date", lambda x: x.isna().sum()),
        missing_customer=("order_delivered_customer_date", lambda x: x.isna().sum())
    )
)

missing_by_status

Distribusi status pesanan:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


,total_orders,missing_approved,missing_carrier,missing_customer
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


#### key findings:
1. Some orders with a delivered status have missing values in the approved_date, delivered_carrier_date, and delivered_customer_date columns. We need to investigate this further.
2. Missing values with other order statuses can happen naturally and are valid.

In [12]:
# Mengambil salinan data orders
orders = datasets["olist_orders_dataset"].copy()

# Kolom tanggal
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Cek pesanan delivered yang memiliki missing values
delivered_missing = orders[
    (orders["order_status"] == "delivered") &
    ((orders["order_delivered_customer_date"].isna()) | (orders["order_delivered_carrier_date"].isna()) | (orders["order_approved_at"].isna()))
]

delivered_missing[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19
26800,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58
39334,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23


#### Marking down orders with missing dates

In [13]:
orders["missing_approved_flag"] = (
    orders["order_approved_at"].isna()
)

orders["missing_carrier_flag"] = (
    orders["order_delivered_carrier_date"].isna()
)

orders["missing_customer_flag"] = (
    orders["order_delivered_customer_date"].isna()
)

In [22]:
datasets["olist_orders_dataset"] = orders

### **3.3 Handle missing values in product table**

In [14]:
datasets["olist_products_dataset"].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [15]:
products = datasets["olist_products_dataset"]

missing_products = products.isna().sum()
missing_products = missing_products[missing_products > 0]

print("Missing values pada produk:")
print(missing_products)

# Cek apakah 610 produk yang tidak memiliki kategori
print(
    "Produk tanpa kategori:",
    products["product_category_name"].isna().sum()
)

Missing values pada produk:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Produk tanpa kategori: 610


#### Missing values imputation

In [16]:
products = datasets["olist_products_dataset"].copy()

# Mengisi kategori yang kosong
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

In [17]:
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

for col in numeric_cols:
    products[col] = products[col].fillna(products[col].median())

In [18]:
missing_dimensions = [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"]

products[missing_dimensions] = products[missing_dimensions].fillna(0)

In [19]:
products.isna().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [29]:
datasets["olist_products_dataset"] = products

### **3.4 Handle Inconsistent Data Types**

In [23]:
# ── Kolom datetime ──────────────────────────────────────────
datetime_map = {
    "olist_orders_dataset": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "olist_order_items_dataset": [
        "shipping_limit_date"
    ],
    "olist_order_reviews_dataset": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table_name, cols in datetime_map.items():
    df = datasets[table_name].copy()
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    datasets[table_name] = df
    print(f"{table_name}: datetime conversion done")

olist_orders_dataset: datetime conversion done
olist_order_items_dataset: datetime conversion done
olist_order_reviews_dataset: datetime conversion done


In [25]:
# ── Kolom kode pos: pastikan string 5 digit ─────────────────
# Kode pos Brasil bisa diawali angka 0, sehingga kalau terbaca
# sebagai integer, leading zero akan hilang (01037 → 1037)

zip_map = {
    "olist_customers_dataset": "customer_zip_code_prefix",
    "olist_sellers_dataset": "seller_zip_code_prefix",
    "olist_geolocation_dataset": "geolocation_zip_code_prefix"
}

for table_name, col in zip_map.items():
    df = datasets[table_name].copy()
    df[col] = df[col].astype(str).str.zfill(5)
    datasets[table_name] = df
    print(f"{table_name}.{col}: converted to zero-padded string")

olist_customers_dataset.customer_zip_code_prefix: converted to zero-padded string
olist_sellers_dataset.seller_zip_code_prefix: converted to zero-padded string
olist_geolocation_dataset.geolocation_zip_code_prefix: converted to zero-padded string


In [26]:
# ── Kolom kategori ───────────────────────────────────────────
# order_status dan payment_type lebih efisien sebagai category
# karena nilai uniknya sedikit

cat_map = {
    "olist_orders_dataset": ["order_status"],
    "olist_order_payments_dataset": ["payment_type"]
}

for table_name, cols in cat_map.items():
    df = datasets[table_name].copy()
    for col in cols:
        df[col] = df[col].astype("category")
    datasets[table_name] = df
    print(f"{table_name}: category conversion done")

olist_orders_dataset: category conversion done
olist_order_payments_dataset: category conversion done


In [28]:
int_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

datasets[int_cols] = datasets[int_cols].astype(int)

In [30]:
# ── Validasi ─────────────────────────────────────────────────
check_tables = [
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_order_reviews_dataset",
    "olist_customers_dataset",
    "olist_products_dataset"
]

for table_name in check_tables:
    print(f"\n{'='*50}")
    print(f"TABEL: {table_name}")
    print(datasets[table_name].dtypes)


TABEL: olist_orders_dataset
order_id                                    str
customer_id                                 str
order_status                           category
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
missing_approved_flag                      bool
missing_carrier_flag                       bool
missing_customer_flag                      bool
dtype: object

TABEL: olist_order_items_dataset
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

TABEL: olist_order_reviews_dataset
review_id                             str
order_id                             